# LLM Reasoning Benchmark — Agent Notebook

Starter notebook for exploring and running the multi-layer benchmark pipeline interactively.

**Kernel:** Select **LLM Reasoning Benchmark** (project `.venv`). The setup cell below also adds the venv to `sys.path` if you use a different kernel.

In [3]:
import sys
from pathlib import Path
from dotenv import load_dotenv

In [4]:
load_dotenv(override=True)

True

In [ ]:
from src.benchmark import (
    ANSWERERS,
    EVALUATOR,
    META_EVALUATOR,
    QUESTION_GENERATOR,
    generate_question,
    get_answers,
    run_benchmark,
)
from src.judge import evaluate_responses, meta_evaluate
from src.models import query_anthropic, query_openai
from src.output import save_benchmark_results

## Default configuration

| Role | Model |
|------|-------|
| Question generator | `QUESTION_GENERATOR` |
| Answerers | `ANSWERERS` |
| Evaluator | `EVALUATOR` |
| Meta-evaluator | `META_EVALUATOR` |

In [ ]:
print(f"Question generator: {QUESTION_GENERATOR}")
print(f"Answerers: {ANSWERERS}")
print(f"Evaluator: {EVALUATOR}")
print(f"Meta-evaluator: {META_EVALUATOR}")

## Run the full pipeline

Set `question` to a string to skip generation, or leave `None` to have GPT-5 generate one.
Results are saved under `output/` (keeps the 3 most recent files).

In [ ]:
# Optional: use a fixed question (saves one API call)
COIN_QUESTION = (
    "You are handed one coin chosen uniformly at random from three: Coin A is "
    "double‑headed (always lands heads), Coin B lands heads with probability 3/4, "
    "and Coin C is fair. You may flip the coin up to three times, observing outcomes "
    "as you go; at any point you may stop and name which coin it is. There is no cost "
    "to flipping, and you win if and only if you name the coin correctly. What "
    "stopping rule and final decision rule maximize your probability of being correct, "
    "and what is that maximum probability (give the exact value)?"
)

question = None  # or COIN_QUESTION

results = run_benchmark(
    question=question,
    verbose=True,
    save_output=True,
    output_dir=str(OUTPUT_DIR),
)

results["meta_evaluator_score"]

## Step-by-step (agent-friendly)

Run one stage at a time when you want finer control over the pipeline.

In [ ]:
question = generate_question()
competitors, answers = get_answers(question)
evaluator_response = evaluate_responses(question, competitors, answers)
meta_score = meta_evaluate(question, competitors, answers, evaluator_response)

step_results = {
    "question": question,
    "competitors": competitors,
    "answers": answers,
    "evaluator_response": evaluator_response,
    "meta_evaluator_score": meta_score,
    "config": {
        "question_generator": QUESTION_GENERATOR,
        "answerers": ANSWERERS,
        "evaluator": EVALUATOR,
        "meta_evaluator": META_EVALUATOR,
    },
}

output_file = save_benchmark_results(step_results, output_dir=OUTPUT_DIR)
print(f"Meta-evaluator score: {meta_score}")
print(f"Saved to: {output_file}")